# Phase Portrait Analysis: Angular Steering as Control on SO(2)

This notebook extracts and visualizes the **phase portrait** of LLM activation dynamics in the Angular Steering plane.

**The pendulum analogy:**
- Activation angle `phi_k` at each layer = pendulum angle
- Angular velocity `omega_k = phi_{k+1} - phi_k` = pendulum angular velocity  
- Layers k = 1, ..., L = discrete time steps
- The steering rotation = control torque

**What we're looking for:**
1. Do harmful and harmless prompts occupy distinct regions in phase space (phi, omega)?
2. Is there a **separatrix** (decision boundary) between them?
3. Do the dynamics look like a **damped pendulum** (wide orbits early, convergence late)?
4. Can we define a **behavioral energy** V = 0.5*omega^2 + (1-cos(phi)) that separates safe from unsafe?

In [3]:
import gc
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import display
from matplotlib.colors import Normalize
from sklearn.decomposition import PCA
from tqdm.notebook import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

from utils import (
    add_hooks,
    get_activations_hook,
    get_input_data,
    tokenize_instructions_fn,
)
from phase_portrait import (
    extract_all_layer_activations,
    compute_steering_plane,
    compute_phase_trajectories,
    compute_statistics,
)

%matplotlib inline
plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.size"] = 11

/Users/hungbui/Work_M4/self/llm-activation-control/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
0it [00:00, ?it/s]


## 1. Configuration

In [4]:
# --- Configuration ---
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"  # Change to your model
POSITION = "mid"  # "mid" (before attn) or "post" (before MLP)
N_SAMPLES = 416  # Number of samples per category
BATCH_SIZE = 8
OUTPUT_DIR = Path("./phase_portrait_output")

model_name = MODEL_ID.split("/")[-1]
output_dir = OUTPUT_DIR / model_name
output_dir.mkdir(parents=True, exist_ok=True)
print(f"Model: {MODEL_ID}")
print(f"Output: {output_dir}")

Model: Qwen/Qwen2.5-3B-Instruct
Output: phase_portrait_output/Qwen2.5-3B-Instruct


## 2. Load Model & Data

In [5]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, device_map="auto", torch_dtype=torch.bfloat16,
)
model.eval()

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, padding_side="left")
if not tokenizer.pad_token:
    tokenizer.pad_token = tokenizer.eos_token

num_layers = model.config.num_hidden_layers
print(f"Loaded {model_name} with {num_layers} layers, hidden_dim={model.config.hidden_size}")

KeyboardInterrupt: 

In [ ]:
harmful_train, _ = get_input_data("harmful", "en")
harmless_train, _ = get_input_data("harmless", "en")
harmful_train = harmful_train[:N_SAMPLES]
harmless_train = harmless_train[:N_SAMPLES]
print(f"Harmful: {len(harmful_train)} samples, Harmless: {len(harmless_train)} samples")

## 3. Extract Activations & Compute Steering Plane

In [ ]:
print("Extracting harmful activations...")
harmful_acts = extract_all_layer_activations(model, harmful_train, tokenizer, [POSITION], BATCH_SIZE)
gc.collect(); torch.cuda.empty_cache()

print("Extracting harmless activations...")
harmless_acts = extract_all_layer_activations(model, harmless_train, tokenizer, [POSITION], BATCH_SIZE)
gc.collect(); torch.cuda.empty_cache()

print(f"Extracted {len(harmful_acts)} layer activations per category")

In [ ]:
plane = compute_steering_plane(harmful_acts, harmless_acts)
b1, b2 = plane["b1"], plane["b2"]
print(f"Selected direction: {plane['selected_key']}")
print(f"b1 . b2 = {(b1 @ b2).item():.6f} (should be ~0)")

harmful_traj = compute_phase_trajectories(harmful_acts, b1, b2, POSITION)
harmless_traj = compute_phase_trajectories(harmless_acts, b1, b2, POSITION)
layers = harmful_traj["layers"]
print(f"Trajectories computed: {len(layers)} layers, harmful={harmful_traj['phi'].shape}, harmless={harmless_traj['phi'].shape}")

## 4. Summary Statistics

Key numbers that validate or invalidate the SO(2) / pendulum analogy.

In [ ]:
stats = compute_statistics(harmful_traj, harmless_traj)
stats["model"] = MODEL_ID
stats["n_layers"] = len(layers)

print(f"{'='*60}")
print(f"PHASE PORTRAIT RESULTS — {model_name}")
print(f"{'='*60}")
print(f"Max angle separation: {stats['max_angle_separation']:.4f} rad "
      f"({np.degrees(stats['max_angle_separation']):.1f} deg) at layer {stats['max_separation_layer']}")
print(f"\nAngular velocity:")
print(f"  Harmful:  mean={stats['harmful_omega_mean']:.4f}, std={stats['harmful_omega_std']:.4f}")
print(f"  Harmless: mean={stats['harmless_omega_mean']:.4f}, std={stats['harmless_omega_std']:.4f}")
print(f"\nBehavioral energy (V = 0.5*w^2 + 1-cos(phi)):")
print(f"  Harmful:  mean={stats['harmful_energy_mean']:.4f}, final_layer={stats['harmful_energy_final_mean']:.4f}")
print(f"  Harmless: mean={stats['harmless_energy_mean']:.4f}, final_layer={stats['harmless_energy_final_mean']:.4f}")
print(f"\nSeparatrix crossings (V > 1.0):")
print(f"  Harmful:  {stats['harmful_separatrix_crossings']}/{harmful_traj['phi'].shape[0]}")
print(f"  Harmless: {stats['harmless_separatrix_crossings']}/{harmless_traj['phi'].shape[0]}")

# Save
with open(output_dir / "statistics.json", "w") as f:
    json.dump(stats, f, indent=2)

## 5. Phase Portrait — Colored by Layer Depth

**What to look for:** Wide orbits in early layers (large omega) that spiral inward in later layers = damped pendulum dynamics.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
n_show = 100

for ax, traj, label in [(axes[0], harmful_traj, "Harmful"), (axes[1], harmless_traj, "Harmless")]:
    for i in range(min(traj["phi"].shape[0], n_show)):
        scatter = ax.scatter(
            traj["phi"][i, :-1], traj["omega"][i, :],
            c=layers[:-1], cmap="viridis", s=3, alpha=0.3, vmin=0, vmax=len(layers)-1,
        )
        ax.plot(traj["phi"][i, :-1], traj["omega"][i, :], alpha=0.05, color="gray", lw=0.5)
    ax.set_xlabel("$\\phi$ (activation angle)"); ax.set_ylabel("$\\omega$ (angular velocity)")
    ax.set_title(f"{label} Prompts")
    ax.axhline(0, color="k", lw=0.5, alpha=0.3); ax.axvline(0, color="k", lw=0.5, alpha=0.3)

plt.colorbar(scatter, ax=axes, label="Layer index", shrink=0.8)
fig.suptitle(f"Phase Portrait by Layer Depth — {model_name}", fontsize=15)
plt.tight_layout()
plt.savefig(output_dir / "phase_portrait_by_layer.pdf", dpi=150, bbox_inches="tight")
plt.show()

## 6. Phase Portrait — Harmful vs Harmless Overlay

**What to look for:** Two distinct clusters/trajectories in phase space = the model separates harmful from harmless in SO(2) dynamics. A clear gap between mean trajectories = candidate separatrix.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
n_show = 100

for i in range(min(harmful_traj["phi"].shape[0], n_show)):
    ax.plot(harmful_traj["phi"][i, :-1], harmful_traj["omega"][i, :], alpha=0.08, color="red", lw=0.5)
for i in range(min(harmless_traj["phi"].shape[0], n_show)):
    ax.plot(harmless_traj["phi"][i, :-1], harmless_traj["omega"][i, :], alpha=0.08, color="blue", lw=0.5)

# Mean trajectories
h_phi = harmful_traj["phi"][:, :-1].mean(0); h_omega = harmful_traj["omega"].mean(0)
s_phi = harmless_traj["phi"][:, :-1].mean(0); s_omega = harmless_traj["omega"].mean(0)

ax.plot(h_phi, h_omega, color="red", lw=2.5, label="Harmful (mean)", zorder=5)
ax.plot(s_phi, s_omega, color="blue", lw=2.5, label="Harmless (mean)", zorder=5)

# Mark start (circle) and end (X)
for phi_m, omega_m, c in [(h_phi, h_omega, "red"), (s_phi, s_omega, "blue")]:
    ax.scatter(phi_m[0], omega_m[0], color=c, s=80, marker="o", zorder=6)
    ax.scatter(phi_m[-1], omega_m[-1], color=c, s=80, marker="X", zorder=6)

ax.set_xlabel("$\\phi$ (activation angle)"); ax.set_ylabel("$\\omega$ (angular velocity)")
ax.set_title(f"Phase Portrait — {model_name}"); ax.legend()
ax.axhline(0, color="k", lw=0.5, alpha=0.3); ax.axvline(0, color="k", lw=0.5, alpha=0.3)
plt.tight_layout()
plt.savefig(output_dir / "phase_portrait_overlay.pdf", dpi=150, bbox_inches="tight")
plt.show()

## 7. Angle & Angular Velocity Evolution Across Layers

**What to look for:** 
- **Angle (phi):** Harmful and harmless should diverge across layers (like Angular Steering Fig.4, but as angles instead of scalar projections)
- **Angular velocity (omega):** Should decrease in magnitude toward later layers = damping / convergence

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

# --- Angle evolution ---
ax = axes[0]
n_show = 50
for i in range(min(harmful_traj["phi"].shape[0], n_show)):
    ax.plot(layers, harmful_traj["phi"][i], alpha=0.08, color="red", lw=0.5)
for i in range(min(harmless_traj["phi"].shape[0], n_show)):
    ax.plot(layers, harmless_traj["phi"][i], alpha=0.08, color="blue", lw=0.5)

for traj, color, label in [(harmful_traj, "red", "Harmful"), (harmless_traj, "blue", "Harmless")]:
    mean = traj["phi"].mean(0); std = traj["phi"].std(0)
    ax.plot(layers, mean, color=color, lw=2, label=f"{label} (mean)")
    ax.fill_between(layers, mean - std, mean + std, alpha=0.15, color=color)

ax.set_ylabel("$\\phi$ (activation angle)"); ax.legend(); ax.set_title(f"Angle Evolution — {model_name}")

# --- Angular velocity ---
ax = axes[1]
mid_layers = layers[:-1]
for traj, color, label in [(harmful_traj, "red", "Harmful"), (harmless_traj, "blue", "Harmless")]:
    mean = traj["omega"].mean(0); std = traj["omega"].std(0)
    ax.plot(mid_layers, mean, color=color, lw=2, label=f"{label} (mean)")
    ax.fill_between(mid_layers, mean - std, mean + std, alpha=0.15, color=color)

ax.axhline(0, color="k", lw=0.5, alpha=0.3)
ax.set_xlabel("Layer"); ax.set_ylabel("$\\omega$ (angular velocity)"); ax.legend()
ax.set_title(f"Angular Velocity — {model_name}")

plt.tight_layout()
plt.savefig(output_dir / "angle_omega_evolution.pdf", dpi=150, bbox_inches="tight")
plt.show()

## 8. 2D Trajectory in the Steering Plane

**What to look for:** Mean trajectories of harmful vs harmless should trace distinct paths through the (b1, b2) plane. The trajectory shape reveals whether the dynamics are linear, spiral, or follow more complex patterns.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 9))
n_show = 50

for i in range(min(harmful_traj["c1"].shape[0], n_show)):
    ax.plot(harmful_traj["c1"][i], harmful_traj["c2"][i], alpha=0.06, color="red", lw=0.5)
for i in range(min(harmless_traj["c1"].shape[0], n_show)):
    ax.plot(harmless_traj["c1"][i], harmless_traj["c2"][i], alpha=0.06, color="blue", lw=0.5)

# Mean trajectories colored by layer
for traj, color, label in [(harmful_traj, "red", "Harmful"), (harmless_traj, "blue", "Harmless")]:
    c1m, c2m = traj["c1"].mean(0), traj["c2"].mean(0)
    ax.plot(c1m, c2m, color=color, lw=2.5, label=f"{label} (mean)", zorder=5)
    scatter = ax.scatter(c1m, c2m, c=layers, cmap="viridis", s=25, zorder=6, vmin=0, vmax=len(layers)-1)
    ax.scatter(c1m[0], c2m[0], color="green", s=100, marker="o", zorder=7)
    ax.scatter(c1m[-1], c2m[-1], color="black", s=100, marker="X", zorder=7)

# Axes labels
max_c = max(abs(harmful_traj["c1"].mean(0)).max(), abs(harmless_traj["c1"].mean(0)).max(),
            abs(harmful_traj["c2"].mean(0)).max(), abs(harmless_traj["c2"].mean(0)).max()) * 0.25
ax.annotate("", xy=(max_c, 0), xytext=(0, 0), arrowprops=dict(arrowstyle="->", color="gray", lw=1.5))
ax.annotate("", xy=(0, max_c), xytext=(0, 0), arrowprops=dict(arrowstyle="->", color="gray", lw=1.5))
ax.text(max_c * 1.05, 0, "$b_1$ (refusal)", fontsize=10, color="gray")
ax.text(0, max_c * 1.05, "$b_2$ (PCA)", fontsize=10, color="gray")

plt.colorbar(scatter, label="Layer index", shrink=0.7)
ax.set_xlabel("$b_1$ coordinate"); ax.set_ylabel("$b_2$ coordinate")
ax.set_title(f"Steering Plane Trajectory — {model_name}"); ax.legend()
ax.set_aspect("equal")
plt.tight_layout()
plt.savefig(output_dir / "steering_plane_trajectory.pdf", dpi=150, bbox_inches="tight")
plt.show()

## 9. Behavioral Energy Landscape

**The pendulum energy:** $V(\phi, \omega) = \frac{1}{2} J \omega^2 + \kappa (1 - \cos(\phi - \phi^*))$

**What to look for:**
- Harmful prompts should have higher energy (further from target equilibrium at $\phi^*=0$)
- The separatrix level $V = \kappa$ should separate the two classes
- Energy should decrease across layers in later layers = damping = the model "settling" into its behavior

In [ ]:
kappa, J = 1.0, 1.0
mid_layers = layers[:-1]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, traj, label, color in [
    (axes[0], harmful_traj, "Harmful", "red"),
    (axes[1], harmless_traj, "Harmless", "blue"),
]:
    phi = traj["phi"][:, :-1]; omega = traj["omega"]
    kinetic = 0.5 * J * omega**2
    potential = kappa * (1 - np.cos(phi))
    energy = kinetic + potential

    mean_e = energy.mean(0); std_e = energy.std(0)
    mean_k = kinetic.mean(0); mean_p = potential.mean(0)

    ax.plot(mid_layers, mean_e, color=color, lw=2, label=f"Total V")
    ax.plot(mid_layers, mean_k, color=color, lw=1.5, ls="--", alpha=0.7, label=f"Kinetic (0.5*$\\omega^2$)")
    ax.plot(mid_layers, mean_p, color=color, lw=1.5, ls=":", alpha=0.7, label=f"Potential (1-cos$\\phi$)")
    ax.fill_between(mid_layers, mean_e - std_e, mean_e + std_e, alpha=0.15, color=color)
    ax.axhline(kappa, color="orange", lw=1.5, ls="--", label=f"Separatrix ($\\kappa$={kappa})")
    ax.set_xlabel("Layer"); ax.set_ylabel("Energy V"); ax.set_title(f"{label}"); ax.legend(fontsize=9)

fig.suptitle(f"Behavioral Energy — {model_name}", fontsize=15)
plt.tight_layout()
plt.savefig(output_dir / "energy_landscape.pdf", dpi=150, bbox_inches="tight")
plt.show()

## 10. Energy-Based Classification (ROC)

Can the behavioral energy at a single layer reliably distinguish harmful from harmless prompts?

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve

# Compute energy at each layer for all samples
harmful_energy = 0.5 * harmful_traj["omega"]**2 + (1 - np.cos(harmful_traj["phi"][:, :-1]))
harmless_energy = 0.5 * harmless_traj["omega"]**2 + (1 - np.cos(harmless_traj["phi"][:, :-1]))

# Per-layer AUROC: can energy at layer k distinguish harmful vs harmless?
aurocs = []
for k in range(len(mid_layers)):
    scores = np.concatenate([harmful_energy[:, k], harmless_energy[:, k]])
    labels = np.concatenate([np.ones(harmful_energy.shape[0]), np.zeros(harmless_energy.shape[0])])
    aurocs.append(roc_auc_score(labels, scores))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# AUROC across layers
ax = axes[0]
ax.plot(mid_layers, aurocs, color="purple", lw=2)
ax.set_xlabel("Layer"); ax.set_ylabel("AUROC")
ax.set_title("Energy-Based Classification AUROC per Layer")
ax.axhline(0.5, color="gray", ls="--", lw=1, label="Random")
ax.legend()
best_layer = mid_layers[np.argmax(aurocs)]
ax.annotate(f"Best: layer {best_layer}\nAUROC={max(aurocs):.3f}",
            xy=(best_layer, max(aurocs)), fontsize=10,
            arrowprops=dict(arrowstyle="->"), xytext=(best_layer + 3, max(aurocs) - 0.05))

# ROC curve at best layer
ax = axes[1]
best_k = np.argmax(aurocs)
scores = np.concatenate([harmful_energy[:, best_k], harmless_energy[:, best_k]])
labels = np.concatenate([np.ones(harmful_energy.shape[0]), np.zeros(harmless_energy.shape[0])])
fpr, tpr, _ = roc_curve(labels, scores)
ax.plot(fpr, tpr, color="purple", lw=2, label=f"Layer {best_layer} (AUROC={max(aurocs):.3f})")
ax.plot([0,1], [0,1], color="gray", ls="--", lw=1)
ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
ax.set_title(f"ROC Curve at Best Layer ({best_layer})"); ax.legend()

fig.suptitle(f"Energy-Based Safety Classification — {model_name}", fontsize=15)
plt.tight_layout()
plt.savefig(output_dir / "energy_roc.pdf", dpi=150, bbox_inches="tight")
plt.show()

print(f"Best AUROC: {max(aurocs):.4f} at layer {best_layer}")

## 11. Dynamics Model Fitting

Fit candidate dynamics models to the empirical (phi_k, omega_k) data and see which one best describes the layer-to-layer evolution.

| Model | Equation | Pendulum analogue |
|-------|----------|-------------------|
| Linear | $\phi_{k+1} = a_k \phi_k + b_k$ | Linearized small-angle pendulum |
| Damped pendulum | $\phi_{k+1} = \phi_k + \omega_k$, $\omega_{k+1} = \alpha \omega_k - \beta \sin(\phi_k)$ | Full nonlinear damped pendulum |
| Identity + noise | $\phi_{k+1} = \phi_k + \epsilon_k$ | No dynamics (random walk) |

In [ ]:
from scipy.optimize import curve_fit

# Combine harmful + harmless for dynamics fitting
all_phi = np.concatenate([harmful_traj["phi"], harmless_traj["phi"]], axis=0)  # (N_total, L)
all_omega = np.concatenate([harmful_traj["omega"], harmless_traj["omega"]], axis=0)  # (N_total, L-1)
n_total, n_layers_total = all_phi.shape

# --- Model 1: Linear per-layer ---
linear_r2 = []
for k in range(n_layers_total - 1):
    x = all_phi[:, k]
    y = all_phi[:, k + 1]
    coeffs = np.polyfit(x, y, 1)
    y_pred = np.polyval(coeffs, x)
    ss_res = np.sum((y - y_pred) ** 2)
    ss_tot = np.sum((y - y.mean()) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0
    linear_r2.append(r2)

# --- Model 2: Damped pendulum (global fit) ---
# omega_{k+1} = alpha * omega_k - beta * sin(phi_k)
# Flatten across layers and samples
omega_curr = all_omega[:, :-1].flatten()
omega_next = all_omega[:, 1:].flatten()
phi_at_omega = all_phi[:, 1:-1].flatten()

def damped_pendulum(X, alpha, beta):
    omega_k, phi_k = X
    return alpha * omega_k - beta * np.sin(phi_k)

try:
    popt, pcov = curve_fit(damped_pendulum, (omega_curr, phi_at_omega), omega_next,
                           p0=[0.9, 0.01], maxfev=10000)
    alpha_fit, beta_fit = popt
    omega_pred = damped_pendulum((omega_curr, phi_at_omega), *popt)
    ss_res = np.sum((omega_next - omega_pred) ** 2)
    ss_tot = np.sum((omega_next - omega_next.mean()) ** 2)
    pendulum_r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0
    pendulum_fit_ok = True
except Exception as e:
    print(f"Pendulum fit failed: {e}")
    alpha_fit, beta_fit, pendulum_r2 = 0, 0, 0
    pendulum_fit_ok = False

# --- Model 3: Identity (baseline) ---
identity_errors = []
for k in range(n_layers_total - 1):
    err = np.mean((all_phi[:, k + 1] - all_phi[:, k]) ** 2)
    identity_errors.append(err)

# --- Plot ---
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax = axes[0]
ax.plot(mid_layers, linear_r2, color="green", lw=2, label="Linear model R$^2$")
ax.axhline(pendulum_r2, color="orange", lw=2, ls="--", label=f"Damped pendulum R$^2$={pendulum_r2:.4f}")
ax.set_xlabel("Layer"); ax.set_ylabel("R$^2$"); ax.legend()
ax.set_title("Dynamics Model Fit Quality")

ax = axes[1]
ax.plot(mid_layers, identity_errors, color="gray", lw=2, label="Identity MSE ($\\phi_{k+1} = \\phi_k$)")
ax.set_xlabel("Layer"); ax.set_ylabel("MSE"); ax.legend()
ax.set_title("Baseline: Identity Model Error")

fig.suptitle(f"Dynamics Fitting — {model_name}", fontsize=15)
plt.tight_layout()
plt.savefig(output_dir / "dynamics_fit.pdf", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nLinear model: mean R^2 = {np.mean(linear_r2):.4f}")
if pendulum_fit_ok:
    print(f"Damped pendulum: alpha={alpha_fit:.4f}, beta={beta_fit:.6f}, R^2={pendulum_r2:.4f}")
    print(f"  Interpretation: alpha={alpha_fit:.4f} = damping factor (1.0=no damping, <1.0=damped)")
    print(f"  Interpretation: beta={beta_fit:.6f} = restoring force ('gravity')")

## 12. Save Trajectories & Cleanup

In [ ]:
np.savez_compressed(
    output_dir / "trajectories.npz",
    harmful_phi=harmful_traj["phi"],
    harmful_omega=harmful_traj["omega"],
    harmful_r=harmful_traj["r"],
    harmful_c1=harmful_traj["c1"],
    harmful_c2=harmful_traj["c2"],
    harmless_phi=harmless_traj["phi"],
    harmless_omega=harmless_traj["omega"],
    harmless_r=harmless_traj["r"],
    harmless_c1=harmless_traj["c1"],
    harmless_c2=harmless_traj["c2"],
    layers=np.array(layers),
    b1=b1.cpu().numpy(),
    b2=b2.cpu().numpy(),
)
print(f"Trajectories saved to {output_dir / 'trajectories.npz'}")
print(f"Statistics saved to {output_dir / 'statistics.json'}")
print(f"All PDFs saved to {output_dir}")